## Stateful operations — joins, dedup, arbitrary state

Three stateful patterns beyond a plain windowed aggregation:

- **Joins.** A **stream-static** join enriches each streaming row from a broadcast dimension table — no watermark needed, all output modes, and the static side is re-read each batch. A **stream-stream** join needs **watermarks on both sides** (so unmatched rows can be evicted) plus a **time-range condition** — without them, state grows forever. `full outer` isn't supported.
- **Deduplication.** Sources are at-least-once, so duplicates happen. `dropDuplicates(["id", "ts"])` **with a watermark** drops re-delivered messages inside the watermark window without unbounded state growth — the watermark is what makes it safe to run forever.
- **Arbitrary per-key state.** For a state machine or custom expiry, reach for **`applyInPandasWithState`** (Spark 3.4+): it receives a key, the new rows for that key, and the current state, and returns updated state + output rows. Pair it with a processing-time or event-time **timeout** to evict inactive keys.